In [34]:
import os
import pickle
import numpy as np
import zipfile
import json
import torch
from os.path import basename


# ----------------------------
# Utility Functions
# ----------------------------

def read_numpy(file_path):
    """
    Read a .npy file and return as numpy array.
    """
    return np.load(file_path)


def load_pkl(file_path):
    """
    Load a pickle file and return its content.
    """
    with open(file_path, 'rb') as f:
        data = pickle.load(f)
    return data


def mmaction2numpy(file_path):
    """
    Convert MMAction2-style pickle output to numpy array.
    Args:
        file_path (str): Path to .pkl file where each entry is a dict with 'pred_score'.
    Returns:
        np.ndarray: Stacked prediction scores, shape [num_samples, num_classes].
    """
    pred_score = []
    pkl_data = load_pkl(file_path)
    for pred_data in pkl_data:
        pred_score.append(pred_data['pred_score'])
    pred_prob = torch.stack(pred_score).numpy()
    return pred_prob


def save_submission(pred_label, name, video_names):
    """
    Save predicted labels to CSV and zip it for submission.
    Args:
        pred_label (np.ndarray): Array of predicted class indices.
        name (str): Output zip name.
        video_names (list): List of video file names (order should match predictions).
    """
    pred_dir = './'
    os.makedirs(pred_dir, exist_ok=True)
    output_csv = os.path.join(pred_dir, 'prediction.csv')
    with open(output_csv, 'w') as f:
        f.write("Id,Target\n")
        for idx, label in enumerate(pred_label):
            f.write(f"{video_names[idx]},{label}\n")

    zip_file_name = os.path.join(pred_dir, f'{name}.zip')
    with zipfile.ZipFile(zip_file_name, "w") as zip_file:
        zip_file.write(output_csv, basename(output_csv))


# ----------------------------
# Load video name list
# ----------------------------

with open('test_list.json') as f:
    video_names = json.load(f)

In [35]:
# ----------------------------
# Model Prediction Probabilities
# ----------------------------

# --- Swin Transformer ---

# Fine-tuned on iMiGUE
swin_small_prob = mmaction2numpy('./2025/swin/swin_small/swin_best_small.pkl')
# MA52 pretrain + iMiGUE fine-tuning
swin_small_trans_prob = mmaction2numpy('./2025/swin/swin_small_trans/swin_last_small_trans.pkl')
# Taylor Stream
swin_small_taylor_prob = mmaction2numpy('./2025/swin/swin_small_taylor/swin_last_small_taylor.pkl')
# Optical Flow
swin_small_of_prob = mmaction2numpy('./2025/swin/swin_base_of2/swin_latest_base_of.pkl')
# Depth
swin_base_depth_prob = mmaction2numpy('./2025/swin/swin_base_depth/swin_best_base_depth.pkl')

# --- P3D / PoseC3D ---

# Joint
p3d_joint_prob = np.array(load_pkl('./2025/p3d/p3d_joint/posec3d_best_joint_sorted.pkl'))
# Limb
p3d_limb_prob = np.array(load_pkl('./2025/p3d/p3d_limb/posec3d_latest_limb_sorted.pkl'))
# RGB + Joint
p3d_rgbJ_11_prob = np.array([d['rgb'] + d['pose'] for d in load_pkl('./2025/p3d/p3d_rgb+j/posec3d_best_rgb+joint_sorted.pkl')])
# RGB + Limb
p3d_rgbL_12_prob = np.array([d['rgb'] + d['pose'] * 2 for d in load_pkl('./2025/p3d/p3d_rgb+l_12/posec3d_latest_rgb+limb_sorted.pkl')])


In [36]:
# ----------------------------
# Ensembling and Submission
# ----------------------------

# Each ensemble can be commented/uncommented as needed for different submissions.
# The prediction label is obtained by np.argmax(probabilities, axis=1).

# Joint only
save_submission(np.argmax(p3d_joint_prob, axis=1), 'output/Joint', video_names)
# Limb only
save_submission(np.argmax(p3d_limb_prob, axis=1), 'output/Limb', video_names)
# Joint + Limb
save_submission(np.argmax(p3d_joint_prob + p3d_limb_prob, axis=1), 'output/Joint_Limb', video_names)
# Swin-based RGB
save_submission(np.argmax(swin_small_trans_prob, axis=1), 'output/RGB', video_names)
# Taylor stream
save_submission(np.argmax(swin_small_taylor_prob, axis=1), 'output/Taylor', video_names)
# Optical Flow
save_submission(np.argmax(swin_small_of_prob, axis=1), 'output/Flow', video_names)
# Depth
save_submission(np.argmax(swin_base_depth_prob, axis=1), 'output/Depth', video_names)
# RGB + Joint (PoseC3D)
save_submission(np.argmax(p3d_rgbJ_11_prob, axis=1), 'output/RGB_J', video_names)
# RGB + Limb (PoseC3D)
save_submission(np.argmax(p3d_rgbL_12_prob, axis=1), 'output/RGB_L', video_names)

# Swin RGB ensemble variants
save_submission(np.argmax(swin_small_prob, axis=1), 'output/RGB(swin)', video_names)
save_submission(np.argmax(swin_small_trans_prob, axis=1), 'output/RGB^(swin)', video_names)
save_submission(np.argmax(swin_small_taylor_prob, axis=1), 'output/Taylor(swin)', video_names)
save_submission(np.argmax(swin_small_of_prob, axis=1), 'output/Flow(swin)', video_names)
save_submission(np.argmax(swin_base_depth_prob, axis=1), 'output/Depth(swin)', video_names)

# Multimodal Ensembles
save_submission(np.argmax(p3d_joint_prob + swin_small_trans_prob, axis=1), 'output/J_RGB^(swin)', video_names)
save_submission(np.argmax(p3d_limb_prob + swin_small_trans_prob, axis=1), 'output/L_RGB^(swin)', video_names)
save_submission(np.argmax(p3d_joint_prob + p3d_limb_prob + swin_small_trans_prob, axis=1), 'output/J_L_RGB^(swin)', video_names)
save_submission(np.argmax(
    p3d_joint_prob + p3d_limb_prob + swin_small_trans_prob + swin_small_taylor_prob, axis=1
), 'output/J_L_RGB^(swin)_Taylor', video_names)
save_submission(np.argmax(
    p3d_joint_prob + p3d_limb_prob + swin_small_trans_prob + swin_small_taylor_prob + swin_small_of_prob, axis=1
), 'output/J_L_RGB^(swin)_Taylor_Flow', video_names)
save_submission(np.argmax(
    p3d_joint_prob + p3d_limb_prob + swin_small_trans_prob + swin_small_taylor_prob + swin_small_of_prob + swin_base_depth_prob, axis=1
), 'output/J_L_RGB^(swin)_Taylor_Flow_Depth', video_names)